# 🌐 BabelNet — Uso com API Key
### Complemento da Aula 1 — Representações Conceituais de Palavras

---

## 📋 Como obter sua API Key gratuita

1. Acesse **https://babelnet.org/**
2. Clique em **Register** e crie sua conta
3. Após confirmar o e-mail, acesse **https://babelnet.org/profile**
4. Copie sua **API Key** (formato: `xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx`)

> ⚠️ A conta gratuita permite **1.000 requisições por dia**.  
> Para uso acadêmico, é possível solicitar um limite maior diretamente no site.

---

## 🔐 Boas práticas com API Key

**Nunca** coloque sua API Key diretamente no código!  
Use um arquivo `.env` para guardar credenciais com segurança.

In [ ]:
# ─────────────────────────────────────────────────────────────
# PASSO 1 — Criar o arquivo .env com sua API Key
#
# Crie um arquivo chamado '.env' na mesma pasta deste notebook
# com o seguinte conteúdo:
#
#   BABELNET_KEY=sua-chave-aqui
#
# A célula abaixo cria esse arquivo automaticamente.
# Substitua o valor pela sua chave real.
# ─────────────────────────────────────────────────────────────

# Criar o arquivo .env (execute apenas uma vez)
with open('.env', 'w') as f:
    f.write('BABELNET_KEY=sua-chave-aqui\n')

print('✅ Arquivo .env criado!')
print('⚠️  Edite o arquivo .env e substitua "sua-chave-aqui" pela sua API Key real.')

In [1]:
# ─────────────────────────────────────────────────────────────
# PASSO 2 — Carregar a API Key do arquivo .env
#
# python-dotenv lê o arquivo .env e carrega as variáveis
# de ambiente sem expor a chave no código
# ─────────────────────────────────────────────────────────────

import os
import requests
from dotenv import load_dotenv

# Carregar as variáveis do arquivo .env
load_dotenv()

# Ler a API Key da variável de ambiente
API_KEY = os.getenv('BABELNET_KEY')

# Verificar se a chave foi carregada
if not API_KEY or API_KEY == 'sua-chave-aqui':
    print('❌ API Key não configurada!')
    print('   Edite o arquivo .env e insira sua chave real.')
else:
    # Mostrar apenas os 8 primeiros caracteres por segurança
    print(f'✅ API Key carregada: {API_KEY[:8]}...')

✅ API Key carregada: 66ebdb15...


---
## 🔍 Exemplo 1 — Buscar Synsets de uma Palavra

O endpoint `/getSynsetIds` retorna todos os synsets associados a uma palavra em um idioma específico.

In [2]:
# ─────────────────────────────────────────────────────────────
# Endpoint: getSynsetIds
# Busca os IDs dos synsets de uma palavra em um idioma
#
# Parâmetros:
#   lemma    → a palavra que queremos buscar
#   searchLang → idioma da palavra (PT = Português, EN = Inglês)
#   key      → sua API Key
# ─────────────────────────────────────────────────────────────

def buscar_synsets(palavra, idioma='PT'):
    """
    Busca os synsets de uma palavra no BabelNet.

    Parâmetros:
        palavra : str  → palavra a buscar (ex: 'cachorro')
        idioma  : str  → código do idioma (ex: 'PT', 'EN', 'ES')

    Retorna:
        list → lista de dicionários com os synsets encontrados
    """
    # URL base da API do BabelNet
    url = 'https://babelnet.io/v9/getSynsetIds'

    # Parâmetros da requisição
    params = {
        'lemma'     : palavra,   # Palavra a buscar
        'searchLang': idioma,    # Idioma da palavra
        'key'       : API_KEY    # Chave de autenticação
    }

    # Fazer a requisição GET para a API
    resposta = requests.get(url, params=params)

    # Verificar se a requisição foi bem-sucedida
    if resposta.status_code == 200:
        return resposta.json()   # Retornar o JSON com os synsets
    else:
        print(f'Erro {resposta.status_code}: {resposta.text}')
        return []


# ── Buscar synsets da palavra 'cachorro' em Português
palavra_busca = 'cachorro'
synsets = buscar_synsets(palavra_busca, idioma='PT')

print(f'🔍 Synsets encontrados para "{palavra_busca}": {len(synsets)}\n')

for i, syn in enumerate(synsets):
    print(f'{i+1}. ID     : {syn["id"]}')
    print(f'   Idioma: {syn["pos"]}')
    print()

🔍 Synsets encontrados para "cachorro": 8

1. ID     : bn:00015267n
   Idioma: NOUN

2. ID     : bn:00055491n
   Idioma: NOUN

3. ID     : bn:00065248n
   Idioma: NOUN

4. ID     : bn:00010741n
   Idioma: NOUN

5. ID     : bn:00024245n
   Idioma: NOUN

6. ID     : bn:06913112n
   Idioma: NOUN

7. ID     : bn:00065261n
   Idioma: NOUN

8. ID     : bn:21937745n
   Idioma: NOUN



---
## 🔍 Exemplo 2 — Buscar Detalhes de um Synset

Com o ID do synset, podemos buscar seus detalhes completos:  
definição, categorias, traduções e relações com outros conceitos.

In [3]:
# ─────────────────────────────────────────────────────────────
# Endpoint: getSynset
# Busca os detalhes completos de um synset pelo seu ID
#
# Parâmetros:
#   id       → ID do synset (ex: 'bn:00015267n')
#   targetLang → idioma desejado para as definições
#   key      → sua API Key
# ─────────────────────────────────────────────────────────────

def buscar_detalhes_synset(synset_id, idioma_alvo='PT'):
    """
    Busca os detalhes de um synset no BabelNet pelo seu ID.

    Parâmetros:
        synset_id   : str → ID do synset (ex: 'bn:00015267n')
        idioma_alvo : str → idioma para as traduções e glosses

    Retorna:
        dict → dicionário com os detalhes do synset
    """
    url = 'https://babelnet.io/v9/getSynset'

    params = {
        'id'        : synset_id,
        'targetLang': idioma_alvo,
        'key'       : API_KEY
    }

    resposta = requests.get(url, params=params)

    if resposta.status_code == 200:
        return resposta.json()
    else:
        print(f'Erro {resposta.status_code}: {resposta.text}')
        return {}


# ── Usar o primeiro synset encontrado no Exemplo 1
if synsets:
    primeiro_id = synsets[0]['id']
    print(f'📋 Buscando detalhes do synset: {primeiro_id}\n')

    detalhes = buscar_detalhes_synset(primeiro_id, idioma_alvo='PT')

    # ── Exibir as definições (glosses) disponíveis
    glosses = detalhes.get('glosses', [])
    print(f'📖 Definições ({len(glosses)} encontradas):')
    for g in glosses[:3]:  # Mostrar até 3 definições
        print(f'   [{g["language"]}] {g["gloss"]}')

    print()

    # ── Exibir os lemas (palavras) no synset por idioma
    senses = detalhes.get('senses', [])
    idiomas_vistos = set()
    print('🌍 Palavras equivalentes em outros idiomas:')
    for sense in senses:
        lang = sense.get('properties', {}).get('language', '')
        lemma = sense.get('properties', {}).get('fullLemma', '')
        if lang and lemma and lang not in idiomas_vistos:
            print(f'   [{lang}] {lemma}')
            idiomas_vistos.add(lang)
        if len(idiomas_vistos) >= 8:  # Limitar a 8 idiomas
            break
else:
    print('⚠️  Nenhum synset encontrado. Verifique sua API Key.')

📋 Buscando detalhes do synset: bn:00015267n

📖 Definições (5 encontradas):
   [PT] Um membro do gênero Canis (provavelmente descende do lobo comum) que tem sido domesticada pelo homem desde os tempos pré-históricos; ocorre em muitas raças.
   [PT] O cão, no Brasil também chamado de cachorro, é um mamífero carnívoro da família dos canídeos, subespécie do lobo, e talvez o mais antigo animal domesticado pelo ser humano.
   [PT] Animal doméstico

🌍 Palavras equivalentes em outros idiomas:
   [PT] cachorro


---
## 🔍 Exemplo 3 — Buscar Relações entre Synsets

O endpoint `/getOutgoingEdges` retorna as relações de um synset com outros conceitos.

In [4]:
# ─────────────────────────────────────────────────────────────
# Endpoint: getOutgoingEdges
# Retorna as relações (arestas) de saída de um synset
# ou seja, com quais outros synsets ele se conecta e como
# ─────────────────────────────────────────────────────────────

def buscar_relacoes(synset_id):
    """
    Busca as relações de um synset com outros conceitos.

    Parâmetros:
        synset_id : str → ID do synset

    Retorna:
        list → lista de relações (arestas)
    """
    url = 'https://babelnet.io/v9/getOutgoingEdges'

    params = {
        'id' : synset_id,
        'key': API_KEY
    }

    resposta = requests.get(url, params=params)

    if resposta.status_code == 200:
        return resposta.json()
    else:
        print(f'Erro {resposta.status_code}: {resposta.text}')
        return []


# ── Buscar relações do synset de 'cachorro'
if synsets:
    primeiro_id = synsets[0]['id']
    relacoes = buscar_relacoes(primeiro_id)

    print(f'🔗 Relações do synset {primeiro_id}: {len(relacoes)} encontradas\n')

    # Agrupar relações por tipo
    tipos_relacao = {}
    for rel in relacoes:
        tipo = rel.get('pointer', {}).get('name', 'UNKNOWN')
        destino = rel.get('target', 'N/A')
        if tipo not in tipos_relacao:
            tipos_relacao[tipo] = []
        tipos_relacao[tipo].append(destino)

    # Exibir relações agrupadas por tipo
    for tipo, destinos in list(tipos_relacao.items())[:6]:  # Top 6 tipos
        print(f'  [{tipo}]')
        for d in destinos[:3]:  # Até 3 destinos por tipo
            print(f'     → {d}')
        print()
else:
    print('⚠️  Nenhum synset disponível. Verifique sua API Key.')

🔗 Relações do synset bn:00015267n: 7227 encontradas

  [member_holonym]
     → bn:00015264n

  [part_meronym]
     → bn:00034966n

  [hyponym]
     → bn:00008602n
     → bn:00008827n
     → bn:00009710n

  [hypernym]
     → bn:00015258n
     → bn:00028151n

  [produced_sound]
     → bn:00008597n
     → bn:00041974n
     → bn:05238087n

  [this_taxon_is_source_of]
     → bn:00238646n
     → bn:02634655n
     → bn:05405134n



---
## 🔍 Exemplo 4 — Pipeline Completo

Combinando os três endpoints anteriores em um único fluxo:  
**palavra → synsets → detalhes → relações**

In [5]:
# ─────────────────────────────────────────────────────────────
# Pipeline completo: dado uma palavra em Português,
# exibe um relatório completo do BabelNet
# ─────────────────────────────────────────────────────────────

def relatorio_babelnet(palavra, idioma='PT'):
    """
    Gera um relatório completo de uma palavra no BabelNet.

    Parâmetros:
        palavra : str → palavra a pesquisar
        idioma  : str → idioma da palavra
    """
    print('=' * 55)
    print(f'  🌐 RELATÓRIO BABELNET — "{palavra.upper()}"')
    print('=' * 55)

    # ── PASSO 1: Buscar synsets
    synsets = buscar_synsets(palavra, idioma)
    if not synsets:
        print('Nenhum synset encontrado.')
        return

    print(f'\n📚 {len(synsets)} synset(s) encontrado(s)')
    print(f'   Usando o primeiro: {synsets[0]["id"]}\n')

    syn_id = synsets[0]['id']

    # ── PASSO 2: Buscar detalhes
    detalhes = buscar_detalhes_synset(syn_id, idioma_alvo=idioma)

    # Definições
    glosses = detalhes.get('glosses', [])
    if glosses:
        print('📖 Definição:')
        print(f'   {glosses[0]["gloss"]}')
        print()

    # Traduções
    senses = detalhes.get('senses', [])
    idiomas_vistos = set()
    traducoes = {}
    for sense in senses:
        props = sense.get('properties', {})
        lang  = props.get('language', '')
        lemma = props.get('fullLemma', '')
        if lang and lemma and lang not in idiomas_vistos:
            traducoes[lang] = lemma
            idiomas_vistos.add(lang)

    if traducoes:
        print('🌍 Traduções:')
        for lang, lemma in list(traducoes.items())[:8]:
            print(f'   [{lang:<3}] {lemma}')
        print()

    # ── PASSO 3: Buscar relações
    relacoes = buscar_relacoes(syn_id)
    if relacoes:
        print(f'🔗 Relações ({len(relacoes)} no total — primeiras 5):')
        for rel in relacoes[:5]:
            tipo    = rel.get('pointer', {}).get('name', 'N/A')
            destino = rel.get('target', 'N/A')
            print(f'   [{tipo}] → {destino}')

    print()
    print('=' * 55)


# ── Executar o pipeline para a palavra 'computador'
relatorio_babelnet('computador', idioma='PT')

  🌐 RELATÓRIO BABELNET — "COMPUTADOR"

📚 3 synset(s) encontrado(s)
   Usando o primeiro: bn:00014691n

📖 Definição:
   Um perito em cálculo (ou no funcionamento de máquinas de calcular).

🌍 Traduções:
   [PT ] computador

🔗 Relações (417 no total — primeiras 5):
   [hyponym] → bn:00001190n
   [hyponym] → bn:00001280n
   [hyponym] → bn:00058293n
   [hyponym] → bn:00075004n
   [derivation] → bn:00084373v



---
## 📋 Referência dos Endpoints da API BabelNet

| Endpoint | Descrição | Parâmetros principais |
|---|---|---|
| `/getSynsetIds` | Lista os IDs dos synsets de uma palavra | `lemma`, `searchLang` |
| `/getSynset` | Detalhes completos de um synset | `id`, `targetLang` |
| `/getOutgoingEdges` | Relações de saída de um synset | `id` |
| `/getSenses` | Lemas/formas de um synset por idioma | `id`, `targetLang` |

> 📖 Documentação completa: **https://babelnet.org/guide**  
> 🔑 Gerenciar sua API Key: **https://babelnet.org/profile**

---

## ⚠️ Limites da conta gratuita

| Plano | Requisições/dia | Uso recomendado |
|---|---|---|
| **Gratuito** | 1.000 | Estudos e experimentos |
| **Acadêmico** | Sob solicitação | Pesquisa universitária |
| **Comercial** | Ilimitado (pago) | Produção |

> 💡 Para solicitar cota acadêmica, envie e-mail para **babelnet@babelnet.org**  
> indicando seu vínculo institucional.